In [1]:
# Imports généraux pour le notebook
import numpy as np
import pandas as pd
import importlib
import librosa
import time
from pathlib import Path

%load_ext autoreload
%autoreload 2

#  modules du projet
from src import config
from src import audio
from src import features



DATA_DIR = Path(r"D:\birdclef_project\data\birdclef-2026")
train_df = pd.read_csv(DATA_DIR / "train.csv")
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"train_df: {len(train_df)} ")
print(f"taxonomy: {len(taxonomy)} ")
print(f"submission : {len(sub.columns) - 1}")
print(f"\ntrain.csv :")
train_df.head()

train_df: 35549 
taxonomy: 234 
submission : 234

train.csv :


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat


In [2]:
# src/ 
import sys
from pathlib import Path
project_root = Path.cwd().parent  
sys.path.insert(0, str(project_root))
print(f"Project root added to sys.path: {project_root}")

Project root added to sys.path: d:\


In [3]:
from src import config

print(f"DATA_DIR exists: {config.DATA_DIR.exists()}")
print(f"AUDIO_TRAIN_DIR exists: {config.AUDIO_TRAIN_DIR.exists()}")
print(f"SR: {config.SR}, N_SAMPLES: {config.N_SAMPLES}")
print(f"MODELS_DIR: {config.MODELS_DIR}")

DATA_DIR exists: True
AUDIO_TRAIN_DIR exists: True
SR: 32000, N_SAMPLES: 160000
MODELS_DIR: D:\birdclef_project\models


In [4]:
import pandas as pd

train_df = pd.read_csv(config.DATA_DIR / "train.csv")
taxonomy = pd.read_csv(config.DATA_DIR / "taxonomy.csv")
sub = pd.read_csv(config.DATA_DIR / "sample_submission.csv")

print(f"train_df: {train_df.shape}")
print(f"  columns: {train_df.columns.tolist()}")
print(f"\ntaxonomy: {taxonomy.shape}")
print(f"  columns: {taxonomy.columns.tolist()}")
print(f"\nsubmission: {sub.shape}")
print(f"  species count: {len(sub.columns) - 1}")

train_df.head()

train_df: (35549, 15)
  columns: ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']

taxonomy: (234, 5)
  columns: ['primary_label', 'inat_taxon_id', 'scientific_name', 'common_name', 'class_name']

submission: (3, 235)
  species count: 234


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat


In [5]:
# Répartition des classes biologiques
print("Classes biologiques dans entraînement:")
print(train_df['class_name'].value_counts())
print(f"\nNombre d'espèces par classe dans taxonomy:")
print(taxonomy['class_name'].value_counts())

Classes biologiques dans entraînement:
class_name
Aves        34799
Amphibia      451
Insecta       199
Mammalia       99
Reptilia        1
Name: count, dtype: int64

Nombre d'espèces par classe dans taxonomy:
class_name
Aves        162
Amphibia     35
Insecta      28
Mammalia      8
Reptilia      1
Name: count, dtype: int64


In [6]:
# Liste  234 espèces 
SPECIES_LIST = sub.columns[1:].tolist()
print(f"Premières espèces: {SPECIES_LIST[:5]}")
print(f"Dernières espèces: {SPECIES_LIST[-5:]}")
print(f"Total: {len(SPECIES_LIST)}")

Premières espèces: ['1161364', '116570', '1176823', '1491113', '1595929']
Dernières espèces: ['yebela1', 'yecmac', 'yecpar', 'yehcar1', 'yeofly1']
Total: 234


In [7]:
# Espèces présentes dans train /soumission
train_species = set(train_df['primary_label'].unique())
sub_species = set(SPECIES_LIST)

n_train_only = len(train_species - sub_species)
n_sub_only = len(sub_species - train_species)
n_overlap = len(train_species & sub_species)

print(f"Espèces dans train: {len(train_species)}")
print(f"Espèces à prédire (submission): {len(sub_species)}")
print(f"couvertes par train: {n_overlap}")
print(f" SANS exemple dans train: {n_sub_only}  zero-shot")
print(f"Espèces dans train mais pas à prédire: {n_train_only}")

Espèces dans train: 206
Espèces à prédire (submission): 234
couvertes par train: 206
 SANS exemple dans train: 28  zero-shot
Espèces dans train mais pas à prédire: 0


In [8]:
# Nombre d'enregistrements par espèce
counts = train_df['primary_label'].value_counts()
print(f"Médiane: {counts.median():.0f} enregistrements/espèce")
print(f"Max: {counts.max()} ({counts.idxmax()})")
print(f"Min: {counts.min()} ({counts.idxmin()})")
print(f"Espèces avec ≤ 5 enregistrements: {(counts <= 5).sum()}")
print(f"Espèces avec ≤ 10 enregistrements: {(counts <= 10).sum()}")
print(f"Espèces avec 1 seul enregistrement: {(counts == 1).sum()}")

Médiane: 125 enregistrements/espèce
Max: 499 (rubthr1)
Min: 1 (116570)
Espèces avec ≤ 5 enregistrements: 18
Espèces avec ≤ 10 enregistrements: 27
Espèces avec 1 seul enregistrement: 4


In [9]:

print("Exemples de filenames:")
print(train_df['filename'].head(5).tolist())

example_file = config.AUDIO_TRAIN_DIR / train_df['filename'].iloc[0]
print(f"\nExemple de chemin: {example_file}")
print(f"Existe: {example_file.exists()}")
print(f"Taille: {example_file.stat().st_size / 1024:.1f} KB" if example_file.exists() else "")

Exemples de filenames:
['1161364/iNat1216197.ogg', '1161364/iNat1114648.ogg', '1161364/iNat810195.ogg', '1161364/iNat818781.ogg', '1161364/iNat556514.ogg']

Exemple de chemin: D:\birdclef_project\data\birdclef-2026\train_audio\1161364\iNat1216197.ogg
Existe: True
Taille: 165.9 KB


In [10]:
#charger les modules 
%load_ext autoreload
%autoreload 2

from src import audio

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import time
import numpy as np

example_file = config.AUDIO_TRAIN_DIR / train_df['filename'].iloc[0]
t0 = time.time()
y_full = audio.load_full_audio(example_file)
print(f"Chargement complet : {time.time()-t0:.2f}s")
print(f"  Forme : {y_full.shape}, durée : {len(y_full)/config.SR:.1f}s")
print(f"  Min/max : {y_full.min():.3f} / {y_full.max():.3f}")


t0 = time.time()
y_win = audio.find_best_window(y_full)
print(f"\nMeilleure fenêtre : {time.time()-t0:.3f}s")
print(f"  Forme : {y_win.shape} (attendu : {config.N_SAMPLES})")
print(f"  RMS : {np.sqrt(np.mean(y_win**2)):.4f}")


center = len(y_full) // 2 - config.N_SAMPLES // 2
y_center = y_full[center:center + config.N_SAMPLES]
print(f"\nRMS fenêtre centrale (ancien comportement) : {np.sqrt(np.mean(y_center**2)):.4f}")
print(f"→ Le ratio best/center confirme que find_best_window trouve plus d'énergie")

y_seg = audio.extract_window_at_offset(y_full, offset_sec=0.0)
print(f"\nSegment à offset=0s : forme {y_seg.shape}")

d:\conda_envs\birdclef\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement complet : 1.66s
  Forme : (576768,), durée : 18.0s
  Min/max : -0.223 / 0.179

Meilleure fenêtre : 0.051s
  Forme : (160000,) (attendu : 160000)
  RMS : 0.0186

RMS fenêtre centrale (ancien comportement) : 0.0170
→ Le ratio best/center confirme que find_best_window trouve plus d'énergie

Segment à offset=0s : forme (160000,)


In [12]:
from src import features


y_win = audio.find_best_window(y_full)

# extraction complète
t0 = time.time()
feat, log_mel = features.extract_handcrafted_features(y_win)
print(f"Extraction : {(time.time()-t0)*1000:.0f} ms")
print(f"Dimension du vecteur : {feat.shape}")
print(f"Forme log_mel : {log_mel.shape}")
print(f"NaN/Inf : {np.isnan(feat).any()}, {np.isinf(feat).any()}")
print(f"Range : [{feat.min():.2f}, {feat.max():.2f}]")

Extraction : 116 ms
Dimension du vecteur : (1126,)
Forme log_mel : (128, 313)
NaN/Inf : False, False
Range : [-476.12, 9554.26]


In [13]:

y_win = audio.find_best_window(y_full)


t0 = time.time()
features.extract_handcrafted_features(y_win)
print(f"Premier appel  : {(time.time()-t0)*1000:.0f} ms")


times = []
for _ in range(5):
    t0 = time.time()
    features.extract_handcrafted_features(y_win)
    times.append((time.time()-t0)*1000)
print(f"Appels 2-6 (ms) : {[f'{t:.0f}' for t in times]}")
print(f"Médiane stabilisée : {np.median(times):.0f} ms")

Premier appel  : 57 ms
Appels 2-6 (ms) : ['56', '57', '59', '57', '57']
Médiane stabilisée : 57 ms


In [14]:

print(f"Top 10 valeurs maximales : {np.sort(feat)[-10:]}")
print(f"Top 10 valeurs minimales : {np.sort(feat)[:10]}")


n_normal = (np.abs(feat) < 100).sum()
print(f"\nDimensions avec |valeur| < 100 : {n_normal} / {len(feat)}")
print(f"Dimensions avec |valeur| > 1000 : {(np.abs(feat) > 1000).sum()}")

Top 10 valeurs maximales : [  20.75086    27.527931   29.430086   32.774765   65.64467    76.87247
  220.02702   370.9639   3533.0635   9554.263   ]
Top 10 valeurs minimales : [-476.117     -75.19948   -74.20191   -73.20122   -65.225235  -63.303417
  -63.187447  -63.094418  -59.480637  -59.12513 ]

Dimensions avec |valeur| < 100 : 1121 / 1126
Dimensions avec |valeur| > 1000 : 2


In [15]:
from src import boaw
import importlib
importlib.reload(boaw)  # au cas où

# mini test sur 20 fichiers 
sample_paths = [
    config.AUDIO_TRAIN_DIR / f
    for f in train_df['filename'].head(20).tolist()
]
print(f"Test sur {len(sample_paths)} fichiers")

#  échantillonner des trames
t0 = time.time()
train_frames = boaw.sample_frames_for_codebook(sample_paths, n_frames_per_clip=5, verbose=False)
print(f"Échantillonnage : {time.time()-t0:.1f}s")
print(f"Forme des trames : {train_frames.shape}  (attendu : ~100 × 128)")

# apprendre les codebooks
t0 = time.time()
boaw_model = boaw.BaggingBoAW(n_codebooks=3, k=64)  # k réduit pour test rapide
boaw_model.fit(train_frames)
print(f"\nApprentissage codebooks : {time.time()-t0:.1f}s")
print(f"Dimension de sortie : {boaw_model.feature_dim}")


y = audio.find_best_window(audio.load_full_audio(sample_paths[0]))
log_mel = features.compute_log_mel(y)

t0 = time.time()
boaw_vec = boaw_model.transform(log_mel)
print(f"\nTransformation d'un clip : {(time.time()-t0)*1000:.1f}ms")
print(f"Vecteur BoAW : {boaw_vec.shape}")
print(f"Somme par codebook (doit valoir ~1.0 × 3) : {boaw_vec.sum():.3f}")
print(f"Range : [{boaw_vec.min():.4f}, {boaw_vec.max():.4f}]")

Test sur 20 fichiers
Échantillonnage : 0.7s
Forme des trames : (100, 128)  (attendu : ~100 × 128)
Apprentissage de 3 codebooks (k=64) sur 100 trames...
  Codebook 1/3 terminé (inertia=98)
  Codebook 2/3 terminé (inertia=221)
  Codebook 3/3 terminé (inertia=161)

Apprentissage codebooks : 0.2s
Dimension de sortie : 192

Transformation d'un clip : 2.0ms
Vecteur BoAW : (192,)
Somme par codebook (doit valoir ~1.0 × 3) : 3.000
Range : [0.0000, 0.4089]


d:\conda_envs\birdclef\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [16]:
from src import nmf_features
importlib.reload(nmf_features)


sample_paths = [config.AUDIO_TRAIN_DIR / f for f in train_df['filename'].head(20).tolist()]


t0 = time.time()
nmf_train_frames = nmf_features.sample_frames_for_nmf(sample_paths, n_frames_per_clip=5, verbose=False)
print(f"Échantillonnage : {time.time()-t0:.1f}s")
print(f"Forme : {nmf_train_frames.shape}")
print(f"Range (doit être [0, 1]) : [{nmf_train_frames.min():.3f}, {nmf_train_frames.max():.3f}]")


t0 = time.time()
nmf_model = nmf_features.NMFFeatureExtractor(n_components=32)  # réduit pour test
nmf_model.fit(nmf_train_frames)
print(f"Apprentissage NMF : {time.time()-t0:.1f}s")


y = audio.find_best_window(audio.load_full_audio(sample_paths[0]))
log_mel = features.compute_log_mel(y)

t0 = time.time()
nmf_vec = nmf_model.transform(log_mel)
print(f"\nTransform : {(time.time()-t0)*1000:.1f}ms")
print(f"Vecteur NMF : {nmf_vec.shape}  (attendu : 3 × 32 = 96)")
print(f"Range : [{nmf_vec.min():.3f}, {nmf_vec.max():.3f}]")
print(f"NaN/Inf : {np.isnan(nmf_vec).any()}, {np.isinf(nmf_vec).any()}")

Échantillonnage : 0.7s
Forme : (100, 128)
Range (doit être [0, 1]) : [0.000, 1.000]
Apprentissage NMF (32 composantes sur 100 trames)...
  Erreur de reconstruction : 3.88
Apprentissage NMF : 0.0s

Transform : 43.0ms
Vecteur NMF : (96,)  (attendu : 3 × 32 = 96)
Range : [0.000, 0.465]
NaN/Inf : False, False


d:\conda_envs\birdclef\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [19]:
from src import feature_pipeline


# Mini test sur 50 fichiers 
test_paths = [config.AUDIO_TRAIN_DIR / f for f in train_df['filename'].head(50).tolist()]
print(f"Test sur {len(test_paths)} fichiers")

t0 = time.time()
boaw_model, nmf_model = feature_pipeline.fit_unsupervised_models(
    test_paths,
    n_codebooks=3,
    boaw_k=64,           
    nmf_components=32,   
    frames_per_clip=5,
)
print(f"\n>>> Modèles non-supervisés ajustés en {time.time()-t0:.1f}s")

Test sur 50 fichiers

=== Apprentissage BoAW + NMF ===
Échantillonnage des trames depuis 50 fichiers...
  Trames BoAW : (250, 128) (2.0s)
  Trames NMF : (250, 128) (1.8s)
Apprentissage de 3 codebooks (k=64) sur 250 trames...
  Codebook 1/3 terminé (inertia=1904)
  Codebook 2/3 terminé (inertia=1490)
  Codebook 3/3 terminé (inertia=1967)
Apprentissage NMF (32 composantes sur 250 trames)...
  Erreur de reconstruction : 8.13

>>> Modèles non-supervisés ajustés en 4.0s


d:\conda_envs\birdclef\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [20]:
from src import feature_pipeline
importlib.reload(feature_pipeline)

# test sur 50 fichiers 
test_paths = [config.AUDIO_TRAIN_DIR / f for f in train_df['filename'].head(50).tolist()]
print(f"Test sur {len(test_paths)} fichiers")


t0 = time.time()
boaw_model, nmf_model = feature_pipeline.fit_unsupervised_models(
    test_paths,
    n_codebooks=3,
    boaw_k=64,           
    nmf_components=32,   
    frames_per_clip=5,
)
print(f"\n>>> Modèles non-supervisés ajustés en {time.time()-t0:.1f}s")

Test sur 50 fichiers

=== Apprentissage BoAW + NMF ===
Échantillonnage des trames depuis 50 fichiers...
  Trames BoAW : (250, 128) (1.8s)
  Trames NMF : (250, 128) (1.8s)
Apprentissage de 3 codebooks (k=64) sur 250 trames...
  Codebook 1/3 terminé (inertia=1904)
  Codebook 2/3 terminé (inertia=1490)
  Codebook 3/3 terminé (inertia=1967)
Apprentissage NMF (32 composantes sur 250 trames)...
  Erreur de reconstruction : 8.13

>>> Modèles non-supervisés ajustés en 3.7s


d:\conda_envs\birdclef\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [21]:
# extraction parallèle
t0 = time.time()
X, valid_mask = feature_pipeline.extract_features_parallel(
    test_paths, boaw_model, nmf_model, n_jobs=6, verbose=True
)
elapsed_parallel = time.time() - t0

# Comparaison 
print("\n--- Comparaison séquentielle (n_jobs=1) ---")
t0 = time.time()
X_seq, _ = feature_pipeline.extract_features_parallel(
    test_paths, boaw_model, nmf_model, n_jobs=1, verbose=False
)
elapsed_seq = time.time() - t0

print(f"\nParallèle (6 workers) : {elapsed_parallel:.1f}s")
print(f"Séquentiel (1 worker) : {elapsed_seq:.1f}s")
print(f"Accélération : {elapsed_seq / elapsed_parallel:.1f}x")
print(f"\nForme de X : {X.shape}  (attendu : (50, 1126 + 192 + 96) = (50, 1414))")



=== Extraction parallèle sur 50 fichiers (6 workers) ===


[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    4.2s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    4.5s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:    4.7s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:    4.9s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:    5.1s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    5.3s
[Parallel(n_jobs=6)]: Done  45 out of  50 | elapsed:    5.4s remaining:    0.5s
[Parallel(n_jobs=6)]: Done  50 out of  50 | elapsed:    5.6s finished



  Terminé en 5.7s (113.4 ms/clip)
  Réussites : 50 / 50
  Échecs : 0
  Matrice résultante : (50, 1414)

--- Comparaison séquentielle (n_jobs=1) ---

Parallèle (6 workers) : 5.7s
Séquentiel (1 worker) : 5.1s
Accélération : 0.9x

Forme de X : (50, 1414)  (attendu : (50, 1126 + 192 + 96) = (50, 1414))


In [22]:
# 300 fichiers
test_paths_300 = [config.AUDIO_TRAIN_DIR / f for f in train_df['filename'].head(300).tolist()]

print("--- Parallèle (6 workers) ---")
t0 = time.time()
X300_par, _ = feature_pipeline.extract_features_parallel(
    test_paths_300, boaw_model, nmf_model, n_jobs=6, verbose=False
)
t_par = time.time() - t0
print(f"Temps : {t_par:.1f}s ({t_par/300*1000:.0f} ms/clip)")

print("\n--- Séquentiel (1 worker) ---")
t0 = time.time()
X300_seq, _ = feature_pipeline.extract_features_parallel(
    test_paths_300, boaw_model, nmf_model, n_jobs=1, verbose=False
)
t_seq = time.time() - t0
print(f"Temps : {t_seq:.1f}s ({t_seq/300*1000:.0f} ms/clip)")

print(f"\nAccélération : {t_seq/t_par:.2f}x")
print(f"Extrapolation pour 35549 fichiers :")
print(f"  Parallèle : {35549 * t_par / 300 / 60:.1f} min")
print(f"  Séquentiel : {35549 * t_seq / 300 / 60:.1f} min")

--- Parallèle (6 workers) ---
Temps : 7.6s (25 ms/clip)

--- Séquentiel (1 worker) ---
Temps : 31.1s (104 ms/clip)

Accélération : 4.10x
Extrapolation pour 35549 fichiers :
  Parallèle : 15.0 min
  Séquentiel : 61.5 min
